# Mini-GPT From Scratch: Two-Stage Training (Pretrain -> Fine-tune)

**Why two stages?** A ~100-150M parameter transformer trained *only* on ~15k chat examples from random initialization never actually learns English -- it has no general-language signal to draw on. Stage 1 pretrains the model on plain text (next-token prediction, no masking) so it learns grammar, vocabulary and world knowledge. Stage 2 then fine-tunes that pretrained model on the Dolly/OpenAssistant conversation data so it learns the `<User>`/`<Assistant>` chat format. This mirrors how real GPT-2/GPT-3-style models are actually built, just at a much smaller scale, and it's fully from-scratch -- no pretrained weights are downloaded from anywhere.

In [ ]:
!pip -q install torch torchvision torchaudio sentencepiece datasets transformers accelerate safetensors einops tqdm

In [ ]:
import os
import math
import random
import sentencepiece as spm
import numpy as np

from tqdm import tqdm
from datasets import load_dataset

import torch
import torch.nn as nn
import torch.nn.functional as F

from dataclasses import dataclass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)


cuda


In [ ]:
# --- GPU DIAGNOSTICS: pick the right autocast dtype for THIS GPU ---
# This is very likely your main speed bug. bfloat16 only runs fast on Ampere+ GPUs
# (A100, L4, RTX 30xx/40xx -> compute capability 8.x). On a T4 or P100 (capability 7.x,
# the free-tier Colab GPU), bf16 matmuls fall back to slow, non-tensor-core paths.
# float16 + GradScaler is the fast, correct choice on those cards.

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
else:
    gpu_name, cap = "CPU", (0, 0)

if cap[0] >= 8:
    AMP_DTYPE = torch.bfloat16
    USE_SCALER = False
else:
    AMP_DTYPE = torch.float16
    USE_SCALER = True

print(f"GPU: {gpu_name} | compute capability: {cap}")
print(f"Using autocast dtype: {AMP_DTYPE} | GradScaler enabled: {USE_SCALER}")


GPU: Tesla T4 | compute capability: (7, 5)
Using autocast dtype: torch.float16 | GradScaler enabled: True


In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size = 24000        # bumped from 16000 -- tokenizer now covers general English text too, not just chat
    block_size = 256

    n_layer = 10
    n_head = 10
    n_embd = 640
    dropout = 0.1
    expansion_factor = 4
    bias = False
    rope_theta = 10000

    # --- BONUS: uncomment for the full GPT-2-small (124M) config ---
    # n_layer = 12
    # n_head = 12
    # n_embd = 768
    # block_size = 1024        # needs a smaller batch_size to fit ~15GB, e.g. batch_size = 16-24

    batch_size = 64                  # starting point only -- calibrate_batch_size() (below, after the
                                      # model is built) auto-raises this to whatever actually fits your GPU
    gradient_accumulation_steps = 2  # fewer/bigger microbatches, less loop/sync overhead

    weight_decay = 0.1
    eval_iters = 50
    eval_interval = 500

    # --- Stage 1: general-text pretraining (teaches grammar / fluency / facts) ---
    pretrain_iters = 8000
    pretrain_lr = 3e-4
    pretrain_min_lr = 3e-5
    pretrain_warmup = 500

    # --- Stage 2: conversational fine-tuning (teaches the chat format/behaviour) ---
    finetune_iters = 6000
    finetune_lr = 1e-4
    finetune_min_lr = 3e-5
    finetune_warmup = 300


## Stage 1 data -- general-text pretraining corpus

In [ ]:
os.makedirs("data", exist_ok=True)

CHAT_OUTPUT = "data/train.txt"
PRETRAIN_OUTPUT = "data/pretrain.txt"


In [ ]:
# WikiText-103 -- ~103M tokens of clean Wikipedia articles. Well-established academic
# choice (easy to justify in your report).
#
# IMPORTANT: streamed instead of fully loaded -- load_dataset(..., split="train") without
# streaming=True materializes the whole split before you can iterate it, which is what
# crashed your session's ~12.7GB of RAM. Streaming pulls/decodes one example at a time.
#
# Swap-in options if you want a bonus angle for your report:
#   - "roneneldan/TinyStories"      -> famous for producing very coherent generations
#                                       even from tiny models; great if you want fast,
#                                       clearly-coherent demo output
#   - a non-English Wikipedia config (e.g. "20220301.hi", "20220301.de" from the
#     "wikipedia" dataset) -> counts as the multilingual/non-English bonus
#
# If you still run out of RAM after this fix: Runtime > Restart runtime first (clears
# anything left over from the crashed run), then lower MAX_CHARS below, or fall back to
# the much smaller "wikitext-2-raw-v1" config (~2M tokens instead of ~103M).

print("Streaming WikiText-103 (pretraining corpus)...")
wikitext = load_dataset(
    "Salesforce/wikitext",   # "wikitext" (no namespace) is broken on current huggingface_hub/datasets -- HfUriError
    "wikitext-103-raw-v1",
    split="train",
    streaming=True,   # never holds the full split in memory
)

MAX_CHARS = 400_000_000  # ~400MB of text -- plenty for several epochs on this model size; lower if RAM is still tight
written = 0

with open(PRETRAIN_OUTPUT, "w", encoding="utf8") as f:
    for sample in tqdm(wikitext, desc="Streaming WikiText-103"):
        line = sample["text"].strip()
        if len(line) == 0:
            continue
        f.write(line + "\n")
        written += len(line) + 1
        if written >= MAX_CHARS:
            break

print(f"Pretraining corpus written to {PRETRAIN_OUTPUT} ({written:,} characters)")


Streaming WikiText-103 (pretraining corpus)...


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Streaming WikiText-103: 1345257it [01:53, 11800.83it/s]

Pretraining corpus written to data/pretrain.txt (400,000,346 characters)


## Stage 2 data -- conversational corpus (Dolly + OpenAssistant)

In [ ]:
def write_conv(f,turns):

    for speaker,text in turns:

        text=text.strip()

        if len(text)==0:
            continue

        f.write(f"<{speaker}>: {text}\n")

    f.write("\n")


In [ ]:
print("Downloading Dolly...")

dolly=load_dataset(
    "databricks/databricks-dolly-15k",
    split="train"
)

print("Downloading OpenAssistant...")

oasst=load_dataset(
    "OpenAssistant/oasst1",
    split="train"
)


README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/10.2k [00:00<?, ?B/s]

data/train-00000-of-00001-b42a775f407cee(…):   0%|          | 0.00/39.5M [00:00<?, ?B/s]

data/validation-00000-of-00001-134b8fd0c(…):   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4401 [00:00<?, ? examples/s]

In [ ]:
with open(
    CHAT_OUTPUT,
    "w",
    encoding="utf8"
) as f:

    for sample in tqdm(dolly):

        user=sample["instruction"]

        if sample["context"].strip():

            user+="\n"+sample["context"]

        assistant=sample["response"]

        write_conv(
            f,
            [
                ("User",user),
                ("Assistant",assistant)
            ]
        )

    conv=[]

    for sample in tqdm(oasst):

        if sample["role"]=="prompter":

            conv.append(
                (
                    "User",
                    sample["text"]
                )
            )

        elif sample["role"]=="assistant":

            conv.append(
                (
                    "Assistant",
                    sample["text"]
                )
            )

        if len(conv)>=2:

            write_conv(
                f,
                conv
            )

            conv=[]


100%|██████████| 84437/84437 [00:20<00:00, 4073.07it/s]


## Shared tokenizer -- trained on BOTH corpora so it covers general English and the chat tags well

In [ ]:
with open("data/tokenizer_corpus.txt", "w", encoding="utf8") as out:
    for path in [PRETRAIN_OUTPUT, CHAT_OUTPUT]:
        with open(path, "r", encoding="utf8") as f:
            out.write(f.read())
            out.write("\n")

print("Combined tokenizer training corpus written.")


Combined tokenizer training corpus written.


In [ ]:
spm.SentencePieceTrainer.train(

input="data/tokenizer_corpus.txt",

model_prefix="minigpt",

vocab_size=GPTConfig.vocab_size,

model_type="bpe",

character_coverage=1.0,

byte_fallback=True,

shuffle_input_sentence=True,

split_digits=True,

pad_id=0,

unk_id=1,

bos_id=2,

eos_id=3,

user_defined_symbols=[
"<User>",
"<Assistant>"
]
)


In [ ]:
sp=spm.SentencePieceProcessor()

sp.load("minigpt.model")

print(sp.encode("<User>: Hello"))

print(sp.decode(sp.encode("<Assistant>: Hi")))


[15383, 4, 15448, 349, 6640]
<Assistant>: Hi


## Stage 1 dataset -- memmapped token binary + random-crop sampling (nanoGPT-style, no per-example Python overhead)

In [ ]:
import numpy as np

def build_pretrain_bins(text_path, tokenizer, out_dir="data", val_fraction=0.005, chunk_chars=5_000_000):
    # IMPORTANT: this version tokenizes in bounded chunks and streams straight to disk.
    # The previous version did f.read() (whole ~400MB file into one Python string) then
    # tokenizer.encode(text) on all of it at once -- that builds a single Python list with
    # ~100M+ int elements before it ever becomes a numpy array, which easily costs several
    # GB of RAM on top of the raw text. That was what crashed your session, not the memmap
    # step below (memmap only touches the small slices it actually reads).
    os.makedirs(out_dir, exist_ok=True)
    train_path = os.path.join(out_dir, "pretrain_train.bin")
    val_path = os.path.join(out_dir, "pretrain_val.bin")

    if os.path.exists(train_path) and os.path.exists(val_path):
        print("Binary files already exist, skipping re-tokenization.")
        return train_path, val_path

    total_bytes = os.path.getsize(text_path)
    val_bytes = max(1_000_000, int(total_bytes * val_fraction))  # at least ~1MB for a stable val set
    train_bytes = total_bytes - val_bytes

    print(f"Tokenizing pretraining corpus in {chunk_chars:,}-character chunks (keeps RAM flat)...")

    train_tokens = 0
    with open(text_path, "r", encoding="utf8") as f_in, open(train_path, "wb") as f_train:

        pbar = tqdm(total=train_bytes, unit="B", unit_scale=True, desc="Tokenizing (train)")
        read_so_far = 0
        while read_so_far < train_bytes:
            chunk = f_in.read(min(chunk_chars, train_bytes - read_so_far))
            if not chunk:
                break
            ids = np.array(tokenizer.encode(chunk, out_type=int), dtype=np.uint16)
            ids.tofile(f_train)          # append this chunk's tokens straight to disk
            train_tokens += len(ids)
            read_so_far += len(chunk)
            pbar.update(len(chunk))
        pbar.close()

        val_text = f_in.read()  # small tail of the file (~1MB) -- fine to hold in RAM directly

    val_ids = np.array(tokenizer.encode(val_text, out_type=int), dtype=np.uint16)
    val_ids.tofile(val_path)

    print(f"Pretrain corpus tokenized | train {train_tokens:,} tokens | val {len(val_ids):,} tokens")
    return train_path, val_path

pretrain_train_bin, pretrain_val_bin = build_pretrain_bins(PRETRAIN_OUTPUT, sp)


def get_batch(bin_path, block_size, batch_size, device):
    # Re-opening the memmap each call is intentional (avoids a known memmap memory leak) --
    # the OS page cache keeps this cheap.
    data = np.memmap(bin_path, dtype=np.uint16, mode="r")
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    # NOTE: y is the SAME span as x, not shifted by one -- MiniGPT.forward() already
    # does the next-token shift internally (see logits[:, :-1] / targets[:, 1:] below),
    # exactly the same convention your ChatDataset already uses.
    y = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    x = x.pin_memory().to(device, non_blocking=True)
    y = y.pin_memory().to(device, non_blocking=True)
    return x, y


Tokenizing pretraining corpus in 5,000,000-character chunks (keeps RAM flat)...


Tokenizing (train): 100%|██████████| 399M/399M [09:19<00:00, 713kB/s]


Pretrain corpus tokenized | train 96,840,896 tokens | val 306,017 tokens


## Stage 2 dataset -- conversation chunks with user-turn masking (unchanged from your original approach)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

with open("data/train.txt", "r", encoding="utf8") as f:
    corpus = f.read()

conversations = [
    c.strip()
    for c in corpus.split("\n\n")
    if c.strip()
]

print(f"Loaded {len(conversations)} conversations")


class ChatDataset(Dataset):

    def __init__(self, conversations, tokenizer, block_size):
        self.examples = []
        self.block_size = block_size

        user_tok = "<User>:"
        assistant_tok = "<Assistant>:"

        for conv in conversations:
            ids = []
            labels = []
            assistant = False

            for line in conv.split("\n"):
                line = line.strip()
                if not line:
                    continue

                if line.startswith(user_tok):
                    assistant = False
                elif line.startswith(assistant_tok):
                    assistant = True

                encoded = tokenizer.encode(
                    line + "\n",
                    out_type=int
                )

                ids.extend(encoded)

                if assistant:
                    labels.extend(encoded)
                else:
                    labels.extend([-100] * len(encoded))

            if len(ids) < 2:
                continue

            # --- FIXED INDENTATION: Loops inside the __init__ constructor ---
            start = 0
            while start < len(ids):
                end = min(start + block_size, len(ids))
                input_ids = ids[start:end]
                target_ids = labels[start:end]

                # Skip chunks with no assistant tokens
                if all(t == -100 for t in target_ids):
                    start = end
                    continue

                if len(input_ids) < block_size:
                    pad = block_size - len(input_ids)
                    input_ids += [0] * pad
                    target_ids += [-100] * pad

                self.examples.append(
                    (
                        torch.tensor(input_ids, dtype=torch.long),
                        torch.tensor(target_ids, dtype=torch.long)
                    )
                )
                start = end

    # --- FIXED INDENTATION: Correctly exposes class methods ---
    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


# Re-initialize the dataset
train_dataset = ChatDataset(
    conversations,
    sp,
    GPTConfig.block_size
)

# --- OPTIMIZED FOR 2-CORE VIRTUAL MACHINES ---
train_loader = DataLoader(
    train_dataset,
    batch_size=GPTConfig.batch_size,
    shuffle=True,
    pin_memory=True,
    drop_last=True,
    num_workers=2,             # CHANGED FROM 4 TO 2: Matches your system's exact CPU core count
    persistent_workers=True    # Keeps workers alive in RAM so they don't recreate every epoch
)

print("Training Samples:", len(train_dataset))

Loaded 179301 conversations
Training Samples: 67308


In [ ]:
x, y = train_dataset[0]

print("Input :", x.shape)
print("Target:", y.shape)

print("Valid labels :", (y != -100).sum().item())
print("Ignored labels:", (y == -100).sum().item())

Input : torch.Size([256])
Target: torch.Size([256])
Valid labels : 29
Ignored labels: 227


In [ ]:
USER_ID = sp.piece_to_id("<User>")

ASSISTANT_ID = sp.piece_to_id("<Assistant>")


def create_labels(x):

    labels = x.clone()

    assistant = False

    for i in range(x.size(0)):

        for j in range(x.size(1)):

            token = x[i, j].item()

            if token == USER_ID:

                assistant = False

            elif token == ASSISTANT_ID:

                assistant = True

            if not assistant:

                labels[i, j] = -100

    return labels

## Model architecture (RoPE attention + SwiGLU FFN -- already a step beyond vanilla GPT-2's learned positional embeddings + GELU MLP, worth calling out as your architectural-modification bonus)

In [ ]:
class RotaryEmbedding(nn.Module):

    def __init__(self, dim, base=10000):

        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(0, dim, 2).float() / dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

    def forward(self, seq_len, device):

        t = torch.arange(
            seq_len,
            device=device
        ).float()

        freqs = torch.outer(
            t,
            self.inv_freq
        )

        emb = torch.cat(
            (freqs,freqs),
            dim=-1
        )

        return emb.cos(), emb.sin()


def rotate_half(x):

    x1=x[...,:x.shape[-1]//2]

    x2=x[...,x.shape[-1]//2:]

    return torch.cat(
        (-x2,x1),
        dim=-1
    )


def apply_rotary(q,k,cos,sin):

    q=q*cos+rotate_half(q)*sin

    k=k*cos+rotate_half(k)*sin

    return q,k

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self):

        super().__init__()

        self.n_head=GPTConfig.n_head

        self.head_dim=GPTConfig.n_embd//GPTConfig.n_head

        self.qkv=nn.Linear(
            GPTConfig.n_embd,
            GPTConfig.n_embd*3,
            bias=False
        )

        self.proj=nn.Linear(
            GPTConfig.n_embd,
            GPTConfig.n_embd,
            bias=False
        )

        self.dropout=nn.Dropout(
            GPTConfig.dropout
        )

        self.rope=RotaryEmbedding(
            self.head_dim
        )

    def forward(self,x):

        B,T,C=x.shape

        qkv=self.qkv(x)

        q,k,v=qkv.chunk(3,dim=-1)

        q=q.view(
            B,T,self.n_head,self.head_dim
        ).transpose(1,2)

        k=k.view(
            B,T,self.n_head,self.head_dim
        ).transpose(1,2)

        v=v.view(
            B,T,self.n_head,self.head_dim
        ).transpose(1,2)

        cos,sin=self.rope(
            T,
            x.device
        )

        cos=cos.unsqueeze(0).unsqueeze(0)

        sin=sin.unsqueeze(0).unsqueeze(0)

        q,k=apply_rotary(
            q,
            k,
            cos,
            sin
        )

        out=F.scaled_dot_product_attention(

            q,

            k,

            v,

            is_causal=True,

            dropout_p=GPTConfig.dropout if self.training else 0

        )

        out=out.transpose(1,2)

        out=out.reshape(B,T,C)

        return self.proj(out)

In [ ]:
class SwiGLU(nn.Module):

    def __init__(self):

        super().__init__()

        hidden=GPTConfig.n_embd*GPTConfig.expansion_factor

        self.w1=nn.Linear(
            GPTConfig.n_embd,
            hidden,
            bias=False
        )

        self.w2=nn.Linear(
            GPTConfig.n_embd,
            hidden,
            bias=False
        )

        self.w3=nn.Linear(
            hidden,
            GPTConfig.n_embd,
            bias=False
        )

    def forward(self,x):

        return self.w3(

            F.silu(self.w1(x))

            *

            self.w2(x)

        )

In [ ]:
class Block(nn.Module):

    def __init__(self):

        super().__init__()

        self.ln1=nn.LayerNorm(
            GPTConfig.n_embd,
            bias=False
        )

        self.attn=MultiHeadAttention()

        self.ln2=nn.LayerNorm(
            GPTConfig.n_embd,
            bias=False
        )

        self.ffn=SwiGLU()

    def forward(self,x):

        x=x+self.attn(
            self.ln1(x)
        )

        x=x+self.ffn(
            self.ln2(x)
        )

        return x

In [ ]:
class MiniGPT(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding=nn.Embedding(
            GPTConfig.vocab_size,
            GPTConfig.n_embd
        )

        self.dropout=nn.Dropout(
            GPTConfig.dropout
        )

        self.blocks=nn.ModuleList(

            [

                Block()

                for _ in range(GPTConfig.n_layer)

            ]

        )

        self.norm=nn.LayerNorm(
            GPTConfig.n_embd,
            bias=False
        )

        self.lm_head=nn.Linear(

            GPTConfig.n_embd,

            GPTConfig.vocab_size,

            bias=False

        )

        self.embedding.weight=self.lm_head.weight

        self.apply(self._init)

    def _init(self,module):

        if isinstance(module,nn.Linear):

            nn.init.normal_(
                module.weight,
                std=0.02
            )

        elif isinstance(module,nn.Embedding):

            nn.init.normal_(
                module.weight,
                std=0.02
            )

    def forward(

        self,

        idx,

        targets=None

    ):

        x=self.embedding(idx)

        x=self.dropout(x)

        for block in self.blocks:

            x=block(x)

        x=self.norm(x)

        logits=self.lm_head(x)

        loss=None

        if targets is not None:

          logits = logits[:, :-1, :].contiguous()

          targets = targets[:, 1:].contiguous()

          loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.view(-1),
            ignore_index=-100
)

        return logits,loss

In [ ]:
model=MiniGPT().to(device)

x=torch.randint(
    0,
    GPTConfig.vocab_size,
    (2,128),
    device=device
)

logits,loss=model(x,x)

print(logits.shape)

print(loss)

print(

sum(

p.numel()

for p in model.parameters()

)/1e6,

"Million Parameters"

)

torch.Size([2, 127, 24000])
tensor(10.2838, device='cuda:0', grad_fn=<NllLossBackward0>)
80.90944 Million Parameters


In [ ]:
## ===================== AUTO-CALIBRATE BATCH SIZE =====================
# Finds the largest batch size that actually fits in your GPU's memory (with a safety
# margin, so a stray longer step or fragmentation later doesn't overflow it and crash
# the session) instead of guessing a fixed number. Uses the REAL model with dummy data;
# gradients are discarded (zero_grad) before any real training happens, so this does not
# affect the model's weights.

def calibrate_batch_size(model, block_size, vocab_size, device, dtype, start_bs, max_bs=512, safety_margin=0.85):
    if device != "cuda":
        print("Not on GPU -- skipping batch size calibration, keeping configured batch_size.")
        return start_bs

    model.train()
    bs = start_bs
    last_good = start_bs

    while bs <= max_bs:
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            x = torch.randint(0, vocab_size, (bs, block_size), device=device)
            y = x.clone()
            with torch.autocast(device_type="cuda", dtype=dtype):
                _, loss = model(x, y)
            loss.backward()
            model.zero_grad(set_to_none=True)
            peak_gb = torch.cuda.max_memory_allocated() / 1e9
            total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"  batch_size={bs:4d} -> peak {peak_gb:.2f} GB / {total_gb:.2f} GB")
            del x, y, loss
            last_good = bs
            bs = int(bs * 1.25) + 1
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if "out of memory" not in str(e).lower():
                raise
            print(f"  batch_size={bs} -> OOM")
            torch.cuda.empty_cache()
            break

    torch.cuda.empty_cache()
    safe_bs = max(8, int(last_good * safety_margin))
    print(f"Max working batch size: {last_good} | using {safe_bs} ({int(safety_margin*100)}% safety margin)")
    return safe_bs


GPTConfig.batch_size = calibrate_batch_size(
    model,
    GPTConfig.block_size,
    GPTConfig.vocab_size,
    device,
    AMP_DTYPE,
    start_bs=GPTConfig.batch_size,
)
print("Final GPTConfig.batch_size:", GPTConfig.batch_size)


  batch_size=  64 -> peak 11.59 GB / 15.64 GB
  batch_size=  81 -> peak 14.53 GB / 15.64 GB
  batch_size=102 -> OOM
Max working batch size: 81 | using 68 (85% safety margin)
Final GPTConfig.batch_size: 68


In [ ]:
# Rebuild the Stage-2 DataLoader now that batch_size may have changed during calibration.
# Also using the RAM headroom you pointed out: prefetch_factor queues more batches ahead
# of time (num_workers stays at 2 -- Colab free/T4 VMs are typically 2 vCPUs, so more
# workers than that just adds contention rather than speed).
train_loader = DataLoader(
    train_dataset,
    batch_size=GPTConfig.batch_size,
    shuffle=True,
    pin_memory=True,
    drop_last=True,
    num_workers=2,
    prefetch_factor=4,
    persistent_workers=True
)
print("Rebuilt train_loader | batch_size =", GPTConfig.batch_size, "| prefetch_factor = 4")


Rebuilt train_loader | batch_size = 68 | prefetch_factor = 4


## Training utilities (shared by both stages)

In [ ]:
from torch.amp import autocast, GradScaler
import math

def build_optimizer(model, lr, weight_decay):
    return torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        betas=(0.9, 0.95),
        weight_decay=weight_decay,
        fused=torch.cuda.is_available()
    )

def build_scheduler(optimizer, warmup_steps, total_steps, lr, min_lr):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        progress = min(progress, 1.0)
        cosine = 0.5 * (1 + math.cos(math.pi * progress))
        min_ratio = min_lr / lr
        return min_ratio + (1 - min_ratio) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scaler = GradScaler(enabled=USE_SCALER)

# torch.compile: solid win on Ampere+/L4/A100, smaller (sometimes negligible) on T4 -- safe either way
try:
    model = torch.compile(model)
    print("torch.compile enabled")
except Exception as e:
    print("torch.compile unavailable, continuing without it:", e)


def safe_forward_backward(x, y, grad_accum_steps):
    """
    Runs forward+backward for one microbatch. If it doesn't fit in GPU memory, frees the
    cache and retries as two half-size microbatches instead of crashing the step (safe
    because LayerNorm is per-sample, so splitting the batch this way is mathematically
    equivalent to running it whole -- it's the same trick gradient accumulation already
    relies on). Returns the (already-divided) loss tensor to add to a running total.
    """
    try:
        with autocast(device_type="cuda", dtype=AMP_DTYPE):
            _, loss = model(x, y)
            loss = loss / grad_accum_steps
        if USE_SCALER:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        return loss.detach()
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if "out of memory" not in str(e).lower():
            raise
        if x.size(0) == 1:
            raise  # can't split further -- a genuine problem, let it surface
        torch.cuda.empty_cache()
        tqdm.write(f"[OOM guard] batch of {x.size(0)} didn't fit -- retrying as two halves")
        mid = x.size(0) // 2
        loss1 = safe_forward_backward(x[:mid], y[:mid], grad_accum_steps)
        loss2 = safe_forward_backward(x[mid:], y[mid:], grad_accum_steps)
        return loss1 + loss2


torch.compile enabled


In [ ]:
@torch.no_grad()
def estimate_pretrain_loss(eval_iters=None):
    eval_iters = eval_iters or GPTConfig.eval_iters
    model.eval()
    losses = []
    for _ in range(eval_iters):
        x, y = get_batch(pretrain_val_bin, GPTConfig.block_size, GPTConfig.batch_size, device)
        with autocast(device_type="cuda", dtype=AMP_DTYPE):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = []
    for i, (x, y) in enumerate(train_loader):
        if i >= GPTConfig.eval_iters:
            break
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with autocast(device_type="cuda", dtype=AMP_DTYPE):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


In [ ]:
os.makedirs("checkpoints", exist_ok=True)


In [ ]:
print("Active device:", device)
print("Is CUDA available?", torch.cuda.is_available())


Active device: cuda
Is CUDA available? True


## Stage 1: general-text pretraining

Standard next-token objective on WikiText-103, no masking. This is the step that teaches the model grammar, fluency and facts. Watch `val loss` / `ppl` (perplexity) -- it should trend steadily down. Checkpoints saved every 1000 steps in case your Colab session disconnects.

In [ ]:
optimizer = build_optimizer(model, lr=GPTConfig.pretrain_lr, weight_decay=GPTConfig.weight_decay)
scheduler = build_scheduler(optimizer, GPTConfig.pretrain_warmup, GPTConfig.pretrain_iters,
                             GPTConfig.pretrain_lr, GPTConfig.pretrain_min_lr)

pbar = tqdm(total=GPTConfig.pretrain_iters, desc="Stage 1: Pretraining")
model.train()
step = 0
optimizer.zero_grad(set_to_none=True)

while step < GPTConfig.pretrain_iters:
    running_loss = torch.zeros((), device=device)

    for micro in range(GPTConfig.gradient_accumulation_steps):
        x, y = get_batch(pretrain_train_bin, GPTConfig.block_size, GPTConfig.batch_size, device)

        running_loss += safe_forward_backward(x, y, GPTConfig.gradient_accumulation_steps)

    if USE_SCALER:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    optimizer.zero_grad(set_to_none=True)
    scheduler.step()

    if step % 20 == 0:
        pbar.set_postfix({
            "loss": f"{running_loss.item() * GPTConfig.gradient_accumulation_steps:.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
        })
    pbar.update(1)

    if step % 200 == 0 and step > 0:
        val_loss = estimate_pretrain_loss()
        tqdm.write(f"[pretrain] step {step} | val loss {val_loss:.4f} | ppl {math.exp(val_loss):.1f}")

    if step % 1000 == 0 and step > 0:
        raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
        torch.save(raw_model.state_dict(), f"checkpoints/pretrain_{step}.pt")

    step += 1

pbar.close()
raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
torch.save(raw_model.state_dict(), "checkpoints/pretrain_final.pt")
print("Stage 1 (pretraining) finished.")


Stage 1: Pretraining:   3%|▎         | 202/8000 [04:50<17:36:01,  8.13s/it, loss=12.5192, lr=1.21e-04]

[pretrain] step 200 | val loss 6.1271 | ppl 458.1


Stage 1: Pretraining:   5%|▌         | 402/8000 [09:09<8:54:49,  4.22s/it, loss=10.4943, lr=2.41e-04]

[pretrain] step 400 | val loss 5.1700 | ppl 175.9


Stage 1: Pretraining:   8%|▊         | 602/8000 [13:34<8:43:40,  4.25s/it, loss=9.6277, lr=3.00e-04]

...

[pretrain] step 7600 | val loss 3.0784 | ppl 21.7


Stage 1: Pretraining:  98%|█████████▊| 7802/8000 [2:54:14<14:11,  4.30s/it, loss=6.0005, lr=3.05e-05]

[pretrain] step 7800 | val loss 3.0854 | ppl 21.9


Stage 1: Pretraining: 100%|██████████| 8000/8000 [2:58:30<00:00,  1.34s/it, loss=5.8633, lr=3.00e-05]


Stage 1 (pretraining) finished.


## Stage 2: conversational fine-tuning

Loads the Stage-1 pretrained weights, then continues training only on the Dolly + OpenAssistant conversation data (with the same user-turn masking you already had). A fresh optimizer/scheduler is used since this is a distinct training phase with its own (lower) learning rate.

In [ ]:
raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
raw_model.load_state_dict(torch.load("checkpoints/pretrain_final.pt", map_location=device))
print("Loaded Stage 1 pretrained weights.")

optimizer = build_optimizer(model, lr=GPTConfig.finetune_lr, weight_decay=GPTConfig.weight_decay)
scheduler = build_scheduler(optimizer, GPTConfig.finetune_warmup, GPTConfig.finetune_iters,
                             GPTConfig.finetune_lr, GPTConfig.finetune_min_lr)

pbar = tqdm(total=GPTConfig.finetune_iters, desc="Stage 2: Fine-tuning")
model.train()
step = 0
optimizer.zero_grad(set_to_none=True)
train_iter = iter(train_loader)

while step < GPTConfig.finetune_iters:
    running_loss = torch.zeros((), device=device)

    for micro in range(GPTConfig.gradient_accumulation_steps):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        running_loss += safe_forward_backward(x, y, GPTConfig.gradient_accumulation_steps)

    if USE_SCALER:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    optimizer.zero_grad(set_to_none=True)
    scheduler.step()

    if step % 20 == 0:
        pbar.set_postfix({
            "loss": f"{running_loss.item() * GPTConfig.gradient_accumulation_steps:.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
        })
    pbar.update(1)

    if step % 100 == 0:
        tqdm.write(f"Step {step} | Loss {running_loss.item() * GPTConfig.gradient_accumulation_steps:.4f}")

    if step > 0 and step % GPTConfig.eval_interval == 0:
        val_loss = estimate_loss()
        tqdm.write(f"[finetune eval] step {step} | val loss {val_loss:.4f}")

    if step % 1000 == 0 and step > 0:
        raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
        torch.save(raw_model.state_dict(), f"checkpoints/finetune_{step}.pt")

    step += 1

pbar.close()
raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
torch.save(raw_model.state_dict(), "checkpoints/final.pt")
print("Stage 2 (fine-tuning) finished.")


Loaded Stage 1 pretrained weights.


Stage 2: Fine-tuning:   0%|          | 1/6000 [00:01<2:18:50,  1.39s/it, loss=14.3119, lr=3.33e-07]

Step 0 | Loss 14.3119


Stage 2: Fine-tuning:   2%|▏         | 101/6000 [01:55<2:27:07,  1.50s/it, loss=9.9222, lr=3.37e-05]

Step 100 | Loss 9.9222


...

Step 5800 | Loss 3.3094


Stage 2: Fine-tuning:  98%|█████████▊| 5901/6000 [1:56:56<02:29,  1.51s/it, loss=3.3767, lr=3.01e-05]

Step 5900 | Loss 3.3767


Stage 2: Fine-tuning: 100%|██████████| 6000/6000 [1:58:50<00:00,  1.19s/it, loss=3.1196, lr=3.00e-05]


Stage 2 (fine-tuning) finished.


In [ ]:
from google.colab import files
import zipfile
import os

# Create an archive containing the model weights and tokenizer
zip_filename = "minigpt_model_export.zip"

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add SentencePiece tokenizer
    if os.path.exists("minigpt.model"):
        zipf.write("minigpt.model")

    # Add final fine-tuned model checkpoint
    if os.path.exists("checkpoints/final.pt"):
        zipf.write("checkpoints/final.pt")

print("Files packaged successfully. Downloading...")
files.download(zip_filename)

Files packaged successfully. Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Target folder in your Google Drive
export_dir = '/content/drive/MyDrive/MiniGPT_Export'
os.makedirs(export_dir, exist_ok=True)

# Copy tokenizer and final checkpoint
shutil.copy("minigpt.model", os.path.join(export_dir, "minigpt.model"))
shutil.copy("checkpoints/final.pt", os.path.join(export_dir, "final.pt"))

print(f"Exported model and tokenizer to: {export_dir}")

## Generation + interactive chat

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=256, temperature=0.8, top_p=0.9, repetition_penalty=1.15):
    model.eval()
    ids = sp.encode(prompt)
    ids = torch.tensor(ids, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        idx = ids[:, -GPTConfig.block_size:]

        # FIXED: Only unpack 2 values here
        logits, _ = model(idx)
        logits = logits[:, -1, :] / temperature

        for token in torch.unique(ids):
            logits[:, token] /= repetition_penalty

        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumulative = torch.cumsum(sorted_probs, dim=-1)

        mask = cumulative > top_p
        mask[..., 1:] = mask[..., :-1].clone()
        mask[..., 0] = False

        sorted_probs[mask] = 0
        sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)

        next_token = torch.multinomial(sorted_probs, 1)
        next_token = sorted_idx.gather(-1, next_token)
        ids = torch.cat([ids, next_token], dim=1)

        if next_token.item() == sp.eos_id():
            break

    return sp.decode(ids[0].tolist())

In [ ]:
MAX_HISTORY_CHARS = 4000  # keep only the recent tail so context never blows past block_size

history = ""

while True:

    user = input("You : ")

    if user.lower() == "exit":
        break

    history += f"<User>: {user}\n<Assistant>: "

    if len(history) > MAX_HISTORY_CHARS:
        history = history[-MAX_HISTORY_CHARS:]

    reply = generate(history)

    assistant = reply[len(history):]

    print("\nAssistant:", assistant, "\n")

    history = reply
